In [ ]:


spark = SparkSession \
    .builder \
    .appName("file-explosion") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", True) \
    .config("spark.hadoop.fs.s3a.fast.upload", True) \
    .config("spark.hadoop.fs.s3a.multipart.size", 104857600) \
    .config("fs.s3a.connection.maximum", 100) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider') \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


spark = (SparkSession 
    .builder 
    .appName("file-explosion") 
    .config("spark.sql.adaptive.enabled", True) 
    .config("spark.sql.files.maxPartitionBytes", "134217728") 
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") 
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") 
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") 
    .config("spark.hadoop.fs.s3a.path.style.access", True) 
    .config("spark.hadoop.fs.s3a.multipart.size", 104857600) 
    .config("spark.hadoop.fs.s3a.connection.maximum", 100)  # prefixo correto
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") 
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") 
    .getOrCreate())

In [13]:
df_iot_data_id = spark.range(0, 100) \
    .select(
      hash('id').alias('id'),
      rand().alias('value'),
      from_unixtime(lit(1701692381 + col('id'))).alias('time')
    )

In [14]:
df_iot_data_id.write.format("parquet").mode("overwrite").partitionBy("id").save("s3a://owshq/iot_id")

In [15]:
df_iot_data_id.createOrReplaceTempView("iot_id")

In [16]:
spark.sql("""
    SELECT * 
    FROM iot_id 
    WHERE id = 519220707
""").show()

+---------+------------------+-------------------+
|       id|             value|               time|
+---------+------------------+-------------------+
|519220707|0.4143272881268887|2023-12-04 12:19:44|
+---------+------------------+-------------------+



In [17]:
spark.sql("""
    SELECT avg(value) 
    FROM iot_id 
    WHERE time >= "2023-12-04 12:19:00" AND time <= "2023-12-04 13:01:20"
""").show()

+-------------------+
|         avg(value)|
+-------------------+
|0.46258968697867997|
+-------------------+



In [18]:
df_iot_data = spark.range(0,50000000, 1, 32) \
    .select(
      hash('id').alias('id'),
      rand().alias('value'),
      from_unixtime(lit(1701692381 + col('id'))).alias('time')
    )

In [19]:
df_iot_data.write.format("parquet").mode("overwrite").save("s3a://owshq/iot/")
df_iot_data.createOrReplaceTempView("iot")

In [20]:
spark.sql("""
    SELECT * 
    FROM iot 
    WHERE id = 519220707
""").show()

+---------+------------------+-------------------+
|       id|             value|               time|
+---------+------------------+-------------------+
|519220707|0.9042335161548656|2023-12-04 12:19:44|
+---------+------------------+-------------------+



In [21]:
spark.sql("""
    SELECT avg(value) 
    FROM iot 
    WHERE time >= "2023-12-04 12:19:00" AND time <= "2023-12-04 13:01:20"
""").show()

+------------------+
|        avg(value)|
+------------------+
|0.5055708704439492|
+------------------+



In [22]:
spark.stop()